In [ ]:
import os
import shutil
import csv
import random

# ================= 參數設定 =================

# 1. 來源圖片資料夾路徑
source_dirs = [
    r"D:\archive\images_001\images",
    r"D:\archive\images_002\images",
    r"D:\archive\images_003\images",
    r"D:\archive\images_004\images",
    r"D:\archive\images_005\images",
    r"D:\archive\images_006\images",
    r"D:\archive\images_007\images",
    r"D:\archive\images_008\images",
    r"D:\archive\images_009\images",
    r"D:\archive\images_010\images",
    r"D:\archive\images_011\images",
    r"D:\archive\images_012\images",
]

# 2. 目標圖片資料夾路徑 (各站點的 images 資料夾)
target_dirs = [
    r"C:\Users\Administrator\Desktop\NIH\site-1\images",
    r"C:\Users\Administrator\Desktop\NIH\site-2\images",
    r"C:\Users\Administrator\Desktop\NIH\site-3\images",
    r"C:\Users\Administrator\Desktop\NIH\site-4\images",
]

# 3. 原始標籤 CSV 總表路徑
MASTER_CSV = r"D:\archive\Data_Entry_2017_processed.csv"

# 支援的圖片副檔名
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp', '.tiff')

# ================= 功能函數 =================

def distribute_images():
    """將來源資料夾的圖片平均搬移到目標資料夾"""
    print("--- 步驟 1：開始檢查並搬移圖片 ---")
    for t_dir in target_dirs:
        os.makedirs(t_dir, exist_ok=True)

    all_images = []
    for s_dir in source_dirs:
        if os.path.exists(s_dir):
            for file in os.listdir(s_dir):
                if file.lower().endswith(IMAGE_EXTENSIONS):
                    all_images.append(os.path.join(s_dir, file))

    total_images = len(all_images)
    if total_images == 0:
        print("來源資料夾沒有找到圖片 (可能已經搬移完畢)。直接進入 CSV 分割步驟。")
        return

    print(f"總共找到 {total_images} 張圖片準備搬移...")
    num_targets = len(target_dirs)
    
    for i, img_path in enumerate(all_images):
        target_index = i % num_targets
        target_folder = target_dirs[target_index]
        file_name = os.path.basename(img_path)
        dest_path = os.path.join(target_folder, file_name)
        
        # 避免檔名重複的保護機制
        if os.path.exists(dest_path):
            base, ext = os.path.splitext(file_name)
            counter = 1
            while os.path.exists(dest_path):
                dest_path = os.path.join(target_folder, f"{base}_{counter}{ext}")
                counter += 1

        shutil.move(img_path, dest_path)
    print("圖片平均分配完畢！\n")


def generate_train_test_csvs():
    """依照各 site 的圖片，從總表抓出對應行數，並平分成 train.csv 和 test.csv"""
    print("--- 步驟 2：開始產生各站點的 train.csv 與 test.csv ---")
    
    # 1. 讀取總表到記憶體 (建立 Dict 以便快速搜尋)
    if not os.path.exists(MASTER_CSV):
        print(f"錯誤：找不到總表 CSV 檔案 -> {MASTER_CSV}")
        return

    master_data = {}
    header = []
    with open(MASTER_CSV, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        header = next(reader) # 讀取標題列
        
        # 預設檔名是第一欄(Index 0) 'Image Index'，讀取所有資料
        for row in reader:
            if row: # 確保不是空行
                img_name = row[0] # NIH 資料集第一欄通常為圖片檔名
                master_data[img_name] = row

    print("已成功讀取總表，開始分配 CSV...")

    # 2. 走訪每個 site 的 images 資料夾
    for target_images_dir in target_dirs:
        site_root_dir = os.path.dirname(target_images_dir) # 回推到上一層，例如 ...\site-1\
        site_name = os.path.basename(site_root_dir)
        
        if not os.path.exists(target_images_dir):
            print(f"找不到資料夾 {target_images_dir}，跳過。")
            continue
            
        # 取得該 site 裡所有的圖片檔名
        images_in_site = [f for f in os.listdir(target_images_dir) if f.lower().endswith(IMAGE_EXTENSIONS)]
        
        # 從總表中找出這些圖片對應的資料列
        site_csv_rows = []
        for img in images_in_site:
            if img in master_data:
                site_csv_rows.append(master_data[img])
            else:
                print(f"警告：圖片 {img} 在 {MASTER_CSV} 中找不到對應的資料！")
        
        # 3. 將資料打亂並平分成兩半
        random.shuffle(site_csv_rows) # 隨機打亂
        mid_point = len(site_csv_rows) // 2 # 找出中點 (除以2並取整數)
        
        train_rows = site_csv_rows[:mid_point]
        test_rows = site_csv_rows[mid_point:]
        
        # 4. 輸出 train.csv
        train_csv_path = os.path.join(site_root_dir, 'train.csv')
        with open(train_csv_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(header)
            writer.writerows(train_rows)
            
        # 5. 輸出 test.csv
        test_csv_path = os.path.join(site_root_dir, 'test.csv')
        with open(test_csv_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(header)
            writer.writerows(test_rows)
            
        print(f"[{site_name}] 完成！分配到 {len(images_in_site)} 張圖片。 train: {len(train_rows)} 筆, test: {len(test_rows)} 筆")

    print("\n所有 CSV 檔案建立完畢！")

# ================= 程式進入點 =================
def main():
    distribute_images()
    generate_train_test_csvs()

if __name__ == "__main__":
    main()

--- 步驟 1：開始檢查並搬移圖片 ---
總共找到 112120 張圖片準備搬移...
